In [24]:
import json
import time
import sys
from pathlib import Path


project_root = Path("../../..").resolve()
sys.path.append(str(project_root))

 
import numpy as np
import pandas as pd
 
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
 
from qiskit.circuit.library import EfficientSU2
from qiskit.primitives import StatevectorEstimator as Estimator
from qiskit_machine_learning.circuit.library import qnn_circuit
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.gradients import SPSAEstimatorGradient
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
    
)

from utils import DataManager

In [10]:
FAST_MODE = True             # True = freeze quantum layer, no quantum gradients (fastest)
USE_SPSA_IF_TRAINING = True  # only matters when FAST_MODE = False
 
NUM_QBITS = 4
ANSATZ_REPS = 1
BATCH_SIZE = 128
NUM_EPOCHS = 50
DECISION_THRESHOLD = 0.5

In [11]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device} (note: quantum layer runs on CPU regardless)")

Using device: cuda (note: quantum layer runs on CPU regardless)


In [12]:
qc, fm_params, anz_params = qnn_circuit(num_qubits=NUM_QBITS, ansatz=EfficientSU2(NUM_QBITS, reps=ANSATZ_REPS))
 
estimator = Estimator()
 
if not FAST_MODE and USE_SPSA_IF_TRAINING:
    gradient = SPSAEstimatorGradient(estimator=estimator, epsilon=0.01)
    estimator_qnn = EstimatorQNN(
        circuit=qc,
        input_params=fm_params,
        weight_params=anz_params,
        estimator=estimator,
        gradient=gradient,
        input_gradients=True,
    )
else:
    estimator_qnn = EstimatorQNN(
        circuit=qc,
        input_params=fm_params,
        weight_params=anz_params,
        estimator=estimator,
        input_gradients=not FAST_MODE,
    )
 
print(f"Quantum layer trainable params: {len(anz_params)}  |  FAST_MODE={FAST_MODE}")

C:\Users\Clent\AppData\Local\Temp\ipykernel_20236\2881831500.py:1: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  qc, fm_params, anz_params = qnn_circuit(num_qubits=NUM_QBITS, ansatz=EfficientSU2(NUM_QBITS, reps=ANSATZ_REPS))
No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


Quantum layer trainable params: 16  |  FAST_MODE=True


In [13]:
df = DataManager('admet/herg').dataset
 
with open(project_root / 'models' / 'random_forest' / 'admet' / 'admet_columns.json', 'r') as f:
    data = json.load(f) 
 
df = df[data['columns']]
df = df.drop(columns=data['herg_exclusion'])
 
X_df = df.drop(columns='label')
y_df = df['label'].astype(int)  


 
X_train, X_test, y_train, y_test = train_test_split(
    X_df, y_df, test_size=0.2, random_state=25, stratify=y_df
)
 
continuous_columns = [c for c in df.columns if not c.startswith("fp_") and c != "label"]
binary_columns = [c for c in df.columns if c.startswith("fp_")]

In [14]:
def scale_features(X_train, X_test=None):
    X_scaler = StandardScaler()
 
    X_train_continuous = pd.DataFrame(
        X_scaler.fit_transform(X_train[continuous_columns]),
        columns=continuous_columns, index=X_train.index
    )
    X_train_binary = X_train[binary_columns].copy()
    X_train_processed = pd.concat([X_train_continuous, X_train_binary], axis=1)
 
    if X_test is not None:
        X_test_continuous = pd.DataFrame(
            X_scaler.transform(X_test[continuous_columns]),
            columns=continuous_columns, index=X_test.index
        )
        X_test_binary = X_test[binary_columns].copy()
        X_test_processed = pd.concat([X_test_continuous, X_test_binary], axis=1)
        return X_train_processed, X_test_processed, X_scaler
 
    return X_train_processed, X_scaler
 
 
X_train_scaled, X_test_scaled, X_scaler = scale_features(X_train, X_test)
 
X_train_tensor = torch.tensor(X_train_scaled.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.to_numpy().reshape(-1, 1), dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.to_numpy().reshape(-1, 1), dtype=torch.float32)
 
train_loader = DataLoader(TensorDataset(X_train_tensor, y_train_tensor), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test_tensor, y_test_tensor), batch_size=BATCH_SIZE, shuffle=False)

In [15]:
class HybridQuantumClassicalClassifier(nn.Module):
    def __init__(self, in_channel, out_channel=1, drop_prob=0.2, hidden_dim=128, fast_mode=True):
        super().__init__()
        self.fast_mode = fast_mode
 
        self.fc1 = nn.Linear(in_channel, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, 64)
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(64)
        self.drop = nn.Dropout(drop_prob)
 
        self.quantum_layer = TorchConnector(estimator_qnn)
        if fast_mode:
            for p in self.quantum_layer.parameters():
                p.requires_grad = False
 
        # NOTE: outputs a raw logit -- no sigmoid here, BCEWithLogitsLoss
        # applies it internally for numerical stability.
        self.classifier = nn.Sequential(
            nn.Linear(64 + estimator_qnn.output_shape[0], 16),
            nn.ReLU(),
            nn.Dropout(drop_prob),
            nn.Linear(16, out_channel)
        )
 
    def forward(self, x):
        x = self.fc1(x)
        x = self.norm1(x)
        x = F.relu(x)
        x = self.drop(x)
 
        x = self.fc2(x)
        x = self.norm2(x)
        x = F.relu(x)
 
        q_input = x[:, :NUM_QBITS]
        if self.fast_mode:
            q_out = self.quantum_layer(q_input.detach())
        else:
            q_out = self.quantum_layer(q_input)
 
        x = torch.cat([x, q_out], dim=1)
        return self.classifier(x)  # logits

In [16]:
model = HybridQuantumClassicalClassifier(
    in_channel=X_train_tensor.shape[1],
    out_channel=1,
    drop_prob=0.2,
    fast_mode=FAST_MODE,
).to(device)
 
# class imbalance handling (optional but recommended): weight positive class
pos_weight_value = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
 
optimizer = torch.optim.Adam(
    [p for p in model.parameters() if p.requires_grad],
    lr=1e-3, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=15)

In [17]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, total_samples = 0.0, 0
    for X_batch, y_batch in loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        bs = X_batch.size(0)
        total_loss += loss.item() * bs
        total_samples += bs
    return total_loss / total_samples
 
 
def evaluate(model, loader, criterion, device, threshold=DECISION_THRESHOLD):
    model.eval()
    total_loss, total_samples = 0.0, 0
    all_probs, all_targets = [], []
    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            bs = X_batch.size(0)
            total_loss += loss.item() * bs
            total_samples += bs
            all_probs.append(torch.sigmoid(logits).cpu())
            all_targets.append(y_batch.cpu())
 
    probs = torch.cat(all_probs, dim=0).numpy().ravel()
    targets = torch.cat(all_targets, dim=0).numpy().ravel()
    preds = (probs >= threshold).astype(int)
 
    acc = accuracy_score(targets, preds)
    prec = precision_score(targets, preds, zero_division=0)
    rec = recall_score(targets, preds, zero_division=0)
    f1 = f1_score(targets, preds, zero_division=0)
    try:
        auc = roc_auc_score(targets, probs)
    except ValueError:
        auc = float("nan")  # only one class present in this split
    cm = confusion_matrix(targets, preds)
 
    return {
        "y_prob": probs, "y_pred": preds,
        "loss": total_loss / total_samples,
        "accuracy": acc, "precision": prec, "recall": rec,
        "f1": f1, "auc": auc, "confusion_matrix": cm,
    }

In [18]:
best_f1 = -np.inf
best_state = None
 
for epoch in range(1, NUM_EPOCHS + 1):
    t0 = time.time()
    print(f"\n{'=' * 70}\nEpoch {epoch}/{NUM_EPOCHS}\n{'=' * 70}", flush=True)
 
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    test_metrics = evaluate(model, test_loader, criterion, device)
    scheduler.step(test_metrics["loss"])
 
    current_f1 = test_metrics["f1"]
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"✓ New best model! F1 = {best_f1:.6f}", flush=True)
 
    dt = time.time() - t0
    print(
        f"\nEpoch [{epoch:03d}/{NUM_EPOCHS}] | Train Loss: {train_loss:.6f} | "
        f"Test Loss: {test_metrics['loss']:.6f} | Acc: {test_metrics['accuracy']:.4f} | "
        f"Prec: {test_metrics['precision']:.4f} | Rec: {test_metrics['recall']:.4f} | "
        f"F1: {current_f1:.4f} | AUC: {test_metrics['auc']:.4f} | Time: {dt:.1f}s",
        flush=True
    )
 
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"\nBest model restored.\nBest Test F1: {best_f1:.6f}", flush=True)
    final_metrics = evaluate(model, test_loader, criterion, device)
    print(f"Confusion matrix:\n{final_metrics['confusion_matrix']}")


Epoch 1/50
✓ New best model! F1 = 0.908108

Epoch [001/50] | Train Loss: 0.423744 | Test Loss: 0.361832 | Acc: 0.8702 | Prec: 0.8842 | Rec: 0.9333 | F1: 0.9081 | AUC: 0.9043 | Time: 2.8s

Epoch 2/50

Epoch [002/50] | Train Loss: 0.363233 | Test Loss: 0.315327 | Acc: 0.8626 | Prec: 0.9000 | Rec: 0.9000 | F1: 0.9000 | AUC: 0.9320 | Time: 1.7s

Epoch 3/50
✓ New best model! F1 = 0.909091

Epoch [003/50] | Train Loss: 0.318098 | Test Loss: 0.283350 | Acc: 0.8779 | Prec: 0.9302 | Rec: 0.8889 | F1: 0.9091 | AUC: 0.9333 | Time: 1.7s

Epoch 4/50

Epoch [004/50] | Train Loss: 0.281283 | Test Loss: 0.258321 | Acc: 0.8626 | Prec: 0.9091 | Rec: 0.8889 | F1: 0.8989 | AUC: 0.9333 | Time: 1.7s

Epoch 5/50
✓ New best model! F1 = 0.916201

Epoch [005/50] | Train Loss: 0.252967 | Test Loss: 0.237991 | Acc: 0.8855 | Prec: 0.9213 | Rec: 0.9111 | F1: 0.9162 | AUC: 0.9363 | Time: 1.7s

Epoch 6/50

Epoch [006/50] | Train Loss: 0.211618 | Test Loss: 0.227135 | Acc: 0.8702 | Prec: 0.9506 | Rec: 0.8556 | F1: 0.

In [26]:
print(classification_report(y_test, test_metrics['y_pred']))

              precision    recall  f1-score   support

           0       0.76      0.78      0.77        41
           1       0.90      0.89      0.89        90

    accuracy                           0.85       131
   macro avg       0.83      0.83      0.83       131
weighted avg       0.86      0.85      0.86       131



In [ ]:
inf_df = DataManager('roselle_dataset', dataset_type='tabular').dataset 

with open('admet_columns_inference.json') as f:
    data = json.load(f)

inf_df = inf_df[data['columns']]

inf_continuous = pd.DataFrame(
    X_scaler.transform(inf_df[continuous_columns]),
    columns=continuous_columns,
    index=inf_df.index
)

inf_binary = inf_df[binary_columns].copy()

inf_df_scaled = pd.concat(
    [inf_continuous, inf_binary],
    axis=1
)


inf_df_tensor = torch.tensor(inf_df_scaled.values, dtype=torch.float32)
inf_loader = DataLoader(inf_df_tensor, batch_size=32, shuffle=False)


In [ ]:
@torch.no_grad()
def predict(model, loader, device="cuda"):
    model.eval()

    predictions = []
    probabilities = []

    for data in loader:
        data = data.to(device)
        logits = model(data)

        probs = torch.sigmoid(logits).squeeze(1)
        pred = (probs >= 0.5).long()

        predictions.extend(pred.cpu().numpy())
        probabilities.extend(probs.cpu().numpy())

    return predictions, probabilities

In [ ]:
predictions, proba = predict(
    model,
    inf_loader,
    device=device
)

pred_df = pd.DataFrame(predictions, columns=['herg'])

In [43]:
final_dataset = pd.read_csv(project_root / 'data' / 'final_dataset' / 'final_roselle_dataset.csv')

final_dataset = pd.concat(objs=[final_dataset, pred_df], axis=1)

output = project_root / 'data' / 'final_dataset'
# output.mkdir(parents=True, exist_ok=True)

output_file = output / 'final_roselle_dataset.csv'
final_dataset.to_csv(output_file, index=False)
# final_dataset
